# Sleep‑EDF 101 – Data Loading & Feature Sketching
**Goal**: Load a polysomnography (PSG) recording and its hypnogram from the Sleep‑EDF dataset, segment the signal into 30‑second epochs, compute canonical EEG band‑power features for each epoch, and preview the resulting feature table.

*Dataset*  `ST7011J0‑PSG.edf` (PSG) and `ST7011JP‑Hypnogram.edf` (annotations)

*Workshop*: AI & Sleep Summer‑School 2025 – 90‑min hands‑on session

## 1 – Imports and environment setup
We import the libraries required for loading EDF files, handling annotations, numerical computations and basic plotting.

In [ ]:
# Uncomment the next line if you have to install dependencies
# %pip install numpy pandas matplotlib mne==1.7.0 neurokit2

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Display pandas tables more compactly
pd.set_option('display.max_columns', None)

## 2 – Load PSG and hypnogram

We **preload** the recording, then keep only the two EEG channels (Fpz-Cz & Pz-Oz) so that all remaining data share the same 100 Hz sampling rate.  
Finally, we attach the hypnogram annotations.

In [ ]:
PSG_PATH = "./data/ST7011J0-PSG.edf"  # ← adjust if needed
HYP_PATH = "./data/ST7011JP-Hypnogram.edf"

# 1) Charger le PSG
raw = mne.io.read_raw_edf(PSG_PATH, preload=True, verbose="error")
raw.pick("eeg")

# 2) Charger **les annotations** correctement
annot = mne.read_annotations(HYP_PATH)
raw.set_annotations(annot)

# Vérification rapide
print(np.unique(annot.description))

## 3 – Epoch the data into 30-second windows

We convert the R&K sleep-stage labels into integer codes, create one event
every 30 s, and build fixed-length epochs.  
`on_missing="ignore"` skips any stage that is absent in this file.

In [ ]:
# Map R&K labels → integer codes
stage_map = {
    "Sleep stage W": 0,
    "Sleep stage 0": 0,      # some files use “0” for Wake
    "Sleep stage 1": 1,
    "Sleep stage 2": 2,
    "Sleep stage 3": 3,
    "Sleep stage 4": 3,      # merge S3 & S4 → N3
    "Sleep stage R": 4,
}

# Create events (one per 30-s chunk)
events, event_id = mne.events_from_annotations(
    raw, event_id=stage_map, chunk_duration=30.0
)

# Drop any event whose code is not in event_id.values()
mask = np.isin(events[:, 2], list(event_id.values()))
events = events[mask]

epochs = mne.Epochs(
    raw,
    events,
    event_id=event_id,
    tmin=0.0,
    tmax=30.0 - 1.0 / raw.info["sfreq"],
    baseline=None,
    preload=True,
    on_missing="ignore",
)

print(epochs)

## 4 – Compute band-power features

We use `epochs.compute_psd(method="welch")`, the modern replacement for the
deprecated `psd_welch`, then average power within canonical frequency bands.

In [ ]:
# Define canonical EEG bands (Hz)
BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 12),
    'sigma': (12, 16),
    'beta':  (16, 30),
}

psd = epochs.compute_psd(
    method="welch",      # multitaper | psd_array_welch | …
    fmin=0.5,
    fmax=30.0,
    n_fft=2048,
    n_overlap=1024,
    picks="eeg",         # already the default here
)
freqs = psd.freqs                       # 1-D array
psds  = psd.get_data(picks=None)        # shape (n_epochs, n_channels, n_freqs)

# Convert to dB
psds_db = 10 * np.log10(psds)

# Average across EEG channels (Fpz‑Cz & Pz‑Oz)
psds_mean = psds_db.mean(axis=1)  # (n_epochs × n_freqs)

# Aggregate band power
band_features = {}
for band, (fmin, fmax) in BANDS.items():
    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
    band_features[f'pow_{band}'] = psds_mean[:, idx].mean(axis=1)

feature_df = pd.DataFrame(band_features)
feature_df['stage'] = epochs.events[:, 2]  # integer stage labels

feature_df.head()

## 5 – Preview and save the feature table
We take a quick look at class balance and persist the table to disk for the next notebook.

In [ ]:
OUT_CSV = "./outputs/sleep_edf_features.csv"
Path(OUT_CSV).parent.mkdir(exist_ok=True)

print(feature_df.stage.value_counts())
feature_df.to_csv(OUT_CSV, index=False)
print(f"Features saved to {OUT_CSV}")

## 6 – Wrap‑up
You have now:
1. Loaded a PSG recording and its hypnogram.
2. Segmented the recording into 30‑second epochs.
3. Computed canonical band‑power features for each epoch.
4. Saved a feature table for downstream modelling.

**Next**: open the *Logistic Regression for Sleep‑Stage Classification* notebook to train a classifier on this table.